# Run16 Simplified MLP Training
Thin orchestration notebook for the Run16 simplified-reward MLP training path. Based on Run13 MLP ablation config with: no execution inertia, simplified reward (base + turnover + terminal only), COVID stress test period, FiLM disabled. Durable logic lives in `src/config.py`, `src/notebook_helpers/tcn_phase1.py`, and the agent/env modules.


## 1) Colab Setup
Clone/sync the repo, clean previous outputs, install requirements, and verify GPU availability.


In [1]:
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

TRAIN_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
TRAIN_REPO_DIR = Path("/content/tcn_tape_vectorized_version_clean")
TRAIN_BRANCH = "feature/run9-alpha-overhaul-20260311"
INSTALL_REQUIREMENTS = True

def run(cmd):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

run(["git", "ls-remote", "--exit-code", "--heads", TRAIN_REPO_URL, TRAIN_BRANCH])

if not (TRAIN_REPO_DIR / ".git").exists():
    run(["git", "clone", "--branch", TRAIN_BRANCH, TRAIN_REPO_URL, str(TRAIN_REPO_DIR)])
else:
    run(["git", "-C", str(TRAIN_REPO_DIR), "fetch", "origin"])
    run(["git", "-C", str(TRAIN_REPO_DIR), "checkout", TRAIN_BRANCH])
    run(["git", "-C", str(TRAIN_REPO_DIR), "reset", "--hard", f"origin/{TRAIN_BRANCH}"])

purge_paths = [
    TRAIN_REPO_DIR / "tcn_fusion_results",
    TRAIN_REPO_DIR / "tcn_results",
    TRAIN_REPO_DIR / "tcn_att_results",
    TRAIN_REPO_DIR / "output_log",
    TRAIN_REPO_DIR / "output_logs",
    TRAIN_REPO_DIR / "data" / "phase1_preparation_artifacts",
    TRAIN_REPO_DIR / "data" / "master_features_NORMALIZED.csv",
    TRAIN_REPO_DIR / "data" / "daily_ohlcv_assets.csv",
    TRAIN_REPO_DIR / "data" / "processed_daily_macro_features.csv",
]

for path in purge_paths:
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    elif path.exists():
        path.unlink()

for cache_dir in TRAIN_REPO_DIR.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)

for ckpt_dir in TRAIN_REPO_DIR.rglob(".ipynb_checkpoints"):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

for mod in list(sys.modules):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]

gc.collect()

os.chdir(TRAIN_REPO_DIR)
if str(TRAIN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_REPO_DIR))

if INSTALL_REQUIREMENTS:
    requirements_file = TRAIN_REPO_DIR / "requirements.txt"
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    run([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)])

print("[OK] Repo synced:", TRAIN_REPO_DIR)
run(["git", "-C", str(TRAIN_REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
run(["git", "-C", str(TRAIN_REPO_DIR), "rev-parse", "HEAD"])
print("[OK] Requirements installed:", INSTALL_REQUIREMENTS)

+ git ls-remote --exit-code --heads https://github.com/Dave-DKings/tcn_tape_vectorized_version.git feature/run9-alpha-overhaul-20260311
+ git clone --branch feature/run9-alpha-overhaul-20260311 https://github.com/Dave-DKings/tcn_tape_vectorized_version.git /content/tcn_tape_vectorized_version_clean
+ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
+ /usr/bin/python3 -m pip install -r /content/tcn_tape_vectorized_version_clean/requirements.txt
[OK] Repo synced: /content/tcn_tape_vectorized_version_clean
+ git -C /content/tcn_tape_vectorized_version_clean rev-parse --abbrev-ref HEAD
+ git -C /content/tcn_tape_vectorized_version_clean rev-parse HEAD
[OK] Requirements installed: True


In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)
if not gpus:
    raise RuntimeError('No GPU visible to TensorFlow. In Colab: Runtime -> Change runtime type -> GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.mixed_precision.set_global_policy('float32')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())


TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">


## 2) Imports
Import the canonical source helpers and training entrypoints.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import build_run13_mlp_config, assert_run13_mlp_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import prepare_phase1_dataset, run_experiment6_tape

RUN_ID = 'run16'
TRAIN_RANDOM_SEED = 42
ANALYSIS_END_DATE = None


## 3) Build Run16 Config and Dataset
Create the Run13 MLP base config, apply Run16 overrides (simplified reward, no inertia, COVID stress split, FiLM off), and prepare the dataset.


In [ ]:
train_config = build_run13_mlp_config('phase1', analysis_end_date=ANALYSIS_END_DATE)
# assert_run13_mlp_config(train_config)  # Skipped — Run16 intentionally diverges

tp = train_config['training_params']
ap = train_config['agent_params']
ppo = ap['ppo_params']
env = train_config['environment_params']
dd = env['drawdown_constraint']

# ==================== RUN 16 OVERRIDES ====================

# --- 1. COVID stress test period ---
from src.config import TRAIN_TEST_SPLIT_DATE_COVID_STRESS
import src.config as _cfg
_cfg.TRAIN_TEST_SPLIT_DATE = TRAIN_TEST_SPLIT_DATE_COVID_STRESS
train_config['TRAIN_TEST_SPLIT_DATE'] = TRAIN_TEST_SPLIT_DATE_COVID_STRESS

# --- 2. Remove execution inertia (beta=1.0 everywhere) ---
tp['action_execution_beta_schedule'] = [{'threshold': 0, 'beta': 1.0}]
tp['action_execution_beta_curriculum'] = {0: 1.0}

# --- 3. Disable FiLM regime conditioning ---
ap['regime_conditioning_enabled'] = False

# --- 4. Simplify reward: KEEP base return + turnover + terminal ---
# Disable DSR/PBRS
env['dsr_scalar'] = 0.0

# Disable benchmark shaping (1/N and SPY)
env['outperformance_bonus_enabled'] = False
env['spy_outperformance_bonus_enabled'] = False

# Disable intra-step TAPE delta
env['intra_step_tape_delta_enabled'] = False

# Disable Lagrangian CVaR
ppo['lagrangian_cvar_penalty_scale'] = 0.0

# Disable drawdown controller
dd['enabled'] = False

# --- 5. Relax checkpoint gate (save checkpoints to evaluate) ---
# Gate on SPY only, not EW (EW gate is too strict for experimentation)
tp['deterministic_validation_require_equal_weight_outperformance'] = False

# --- 6. Disable early stopping completely ---
tp['training_early_stop_enabled'] = False

# --- Guard assertions ---
assert ap['regime_conditioning_enabled'] == False, 'FiLM still on!'
assert env['dsr_scalar'] == 0.0, 'DSR still on!'
assert env['outperformance_bonus_enabled'] == False, 'EW shaping still on!'
assert dd['enabled'] == False, 'Drawdown controller still on!'
assert tp['training_early_stop_enabled'] == False, 'Early stop still on!'

print('[Run16] Simplified config ready.')
print(f'  split_date = {train_config["TRAIN_TEST_SPLIT_DATE"]}')
print(f'  regime_conditioning = {ap["regime_conditioning_enabled"]}')
print(f'  execution_beta = {tp["action_execution_beta_curriculum"]}')
print(f'  dsr_scalar = {env["dsr_scalar"]}')
print(f'  benchmark_shaping_1n = {env["outperformance_bonus_enabled"]}')
print(f'  benchmark_shaping_spy = {env["spy_outperformance_bonus_enabled"]}')
print(f'  intra_step_tape = {env["intra_step_tape_delta_enabled"]}')
print(f'  lagrangian_cvar_scale = {ppo["lagrangian_cvar_penalty_scale"]}')
print(f'  drawdown_controller = {dd["enabled"]}')
print(f'  ew_gate = {tp["deterministic_validation_require_equal_weight_outperformance"]}')
print(f'  early_stop = {tp["training_early_stop_enabled"]}')

# --- Architecture summary (kept from run13) ---
print()
print('[Run16] Architecture (inherited from Run13 MLP base):')
print('  architecture =', ap['actor_critic_type'])
print('  mlp_backbone =', {
    'actor_hidden_dims': ap['actor_hidden_dims'],
    'critic_hidden_dims': ap['critic_hidden_dims'],
    'mlp_dropout': ap.get('mlp_dropout', None),
})
print('  mixture_dirichlet =', {
    'enabled': ap['mixture_dirichlet_enabled'],
    'num_components': ap['mixture_dirichlet_num_components'],
    'eval_mode': ap['mixture_dirichlet_eval_mode'],
})
print('  distributional_critic =', ap['distributional_critic_enabled'])
print('  turnover =', {
    'target': env['target_turnover'],
    'base_scalar': env['turnover_penalty_scalar'],
    'curriculum': tp['turnover_penalty_curriculum'],
})
print('  terminal_bonus =', {
    'mode': env['tape_terminal_bonus_mode'],
    'baseline': env['tape_terminal_baseline'],
    'scalar': env['tape_terminal_scalar'],
    'gate_a': env['tape_terminal_gate_a_enabled'],
})

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir=str(TRAIN_REPO_DIR / 'data_exports'),
)

actuarial_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Actuarial_')]
if actuarial_cols:
    raise RuntimeError(f'Actuarial columns should be absent for Run16: {actuarial_cols}')

alpha_ret_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('AlphaRet_')]
expected_alpha_cols = {'AlphaRet_1d', 'AlphaRet_5d', 'AlphaRet_20d', 'AlphaRet_5d_Z', 'AlphaRet_20d_Z'}
missing_alpha_cols = sorted(list(expected_alpha_cols - set(alpha_ret_cols)))
if missing_alpha_cols:
    raise RuntimeError(f'Missing expected alpha-return columns: {missing_alpha_cols}')

fundamental_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Fundamental_')]
if fundamental_cols:
    raise RuntimeError(f'Fundamental columns should be absent: {fundamental_cols}')

print('[OK] Train shape:', train_phase1_data.train_df.shape)
print('[OK] Test shape:', train_phase1_data.test_df.shape)
print('[OK] Actuarial feature check passed: none present (disabled)')
print('[OK] Alpha-return feature check passed:', sorted(alpha_ret_cols)[:10])
print('[OK] Fundamental feature check passed: none present')


## 4) Run Training
Launch the Run16 simplified MLP training path.


In [5]:
RUN_TRAINING = True

if RUN_TRAINING:
    training_params = train_config['training_params']
    print('[START] Starting training')
    print('Architecture:', train_config['agent_params'].get('actor_critic_type'))
    print('max_total_timesteps:', training_params['max_total_timesteps'])
    print('num_parallel_envs:', training_params.get('num_parallel_envs', 1))

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config['agent_params'].get('actor_critic_type'),
        timesteps_per_update=training_params.get('timesteps_per_ppo_update', 1008),
        max_total_timesteps=training_params['max_total_timesteps'],
    )

    print('[OK] Training complete')
    print('checkpoint_prefix:', train_experiment6.checkpoint_path)
else:
    print('[SKIP] RUN_TRAINING=False')


[START] Starting training
Architecture: MLP
max_total_timesteps: 300000
num_parallel_envs: 4

EXPERIMENT 6: MLP Enhanced + TAPE Three-Component
Architecture: MLP
Results root: /content/tcn_tape_vectorized_version_clean/results
Working dir: /content/tcn_tape_vectorized_version_clean
Covariance Features: Yes
🎯 REWARD SYSTEM: TAPE (Three-Component v3)
   Profile: BalancedGrowth
   Daily: Base + DSR/PBRS + Turnover_Proximity
   Terminal: mode=signed | baseline=0.20 | scalar=10.0 (clipped ±10.0)
   Gate A: enabled (Sharpe <= 0.00 or MDD >= 30.0% -> force non-positive terminal bonus)
   Neutral Band: enabled (±0.020 around baseline)
   [CYCLE] Profile Manager: disabled (static profile only)
[RAND] Experiment Seed: 6042 (Base: 42, Offset: 6000)
[OK] Features: Enhanced (includes 2 covariance eigenvalues)
   Eigenvalues: ['Covariance_Eigenvalue_0', 'Covariance_Eigenvalue_1']
   Train shape: (25170, 67)
   Test shape: (9180, 67)
   ℹ️ Actuarial features disabled by config.

🏗️ Creating THREE-COM

## 5) Inspect Latest Training Logs
Load the latest training CSVs and inspect the current run.


In [ ]:
TRAIN_RESULTS_ROOT = TRAIN_REPO_DIR / 'tcn_fusion_results'
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / 'logs'

episodes_files = sorted(TRAIN_LOGS_DIR.glob('*episodes*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f'No episodes CSV found in {TRAIN_LOGS_DIR}')
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print('Episodes file:', train_episodes_path)
    print('Rows:', len(train_episodes_df))
    display(train_episodes_df.tail(20))

step_diag_files = sorted(TRAIN_LOGS_DIR.glob('*step_diagnostics*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if step_diag_files:
    step_diag_path = step_diag_files[0]
    step_diag_df = pd.read_csv(step_diag_path)
    print('Step diagnostics file:', step_diag_path)
    print('Rows:', len(step_diag_df))
    display(step_diag_df.tail(20))


## 6) Export Artifacts (Optional)
Zip the latest results and optionally copy them to Google Drive.


In [ ]:
import subprocess

EXPORT_RESULTS_ZIP = True
COPY_TO_DRIVE = True

# Dynamic naming: includes architecture type and run ID
_arch = train_config['agent_params'].get('actor_critic_type', 'unknown').lower()
EXPORT_PATH = Path(f'/content/tcn_tape_{_arch}_{RUN_ID}.zip')

# Dedicated Drive folder for all run exports
DRIVE_EXPORT_DIR = Path('/content/drive/MyDrive/tcn_tape_runs')

if EXPORT_RESULTS_ZIP:
    include_paths = [
        TRAIN_REPO_DIR / 'tcn_fusion_results',
        TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
        TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
        TRAIN_REPO_DIR / 'data_exports',
        TRAIN_REPO_DIR / 'output_log',
        TRAIN_REPO_DIR / 'output_logs',
    ]
    existing = []
    seen = set()
    for path in include_paths:
        resolved = path.resolve()
        if resolved.exists() and resolved not in seen:
            seen.add(resolved)
            existing.append(resolved)

    if existing:
        if EXPORT_PATH.exists():
            EXPORT_PATH.unlink()
        relative_items = [str(path.relative_to(TRAIN_REPO_DIR)) for path in existing]
        subprocess.run(
            ['bash', '-lc', 'cd "{}" && zip -qr "{}" {}'.format(TRAIN_REPO_DIR, EXPORT_PATH, ' '.join(f'"{item}"' for item in relative_items))],
            check=True,
        )
        print('[OK] Created:', EXPORT_PATH)

        if COPY_TO_DRIVE:
            from google.colab import drive
            drive.mount('/content/drive')
            DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
            dest = DRIVE_EXPORT_DIR / EXPORT_PATH.name
            subprocess.run(['cp', str(EXPORT_PATH), str(dest)], check=True)
            print('[OK] Copied to Drive:', dest)
    else:
        print('[WARN] Nothing to export.')
else:
    print('[SKIP] EXPORT_RESULTS_ZIP=False')


In [ ]:
from pathlib import Path
import csv, re

root = Path("/content/tcn_tape_vectorized_version_clean/tcn_fusion_results")
log_dir = root / "logs"
ckpt_dir = root / "high_watermark_checkpoints"

# latest episodes csv
ep_csv = sorted(log_dir.glob("*_episodes.csv"), key=lambda p: p.stat().st_mtime)[-1]
print("episodes_csv:", ep_csv)

rows = list(csv.DictReader(ep_csv.open()))
print("rows:", len(rows))
if rows:
    last = rows[-1]
    for k in ["update", "step", "episode", "episode_sharpe", "episode_return_pct", "episode_turnover_pct"]:
        if k in last:
            print(k, "=", last[k])

# best checkpoint by Sharpe tag in filename
pat = re.compile(r"shp([pm]?\d+p\d+)")
best = None
for f in ckpt_dir.glob("*_actor.weights.h5"):
    m = pat.search(f.name)
    if not m:
        continue
    t = m.group(1)
    sign = -1 if t.startswith("m") else 1
    t = t[1:] if t[0] in "pm" else t
    sh = sign * float(t.replace("p", "."))
    if best is None or sh > best[0]:
        best = (sh, f)

print("best_checkpoint:", best[1] if best else None)
print("best_sharpe_tag:", best[0] if best else None)

In [ ]:
# Rank all saved checkpoints by deterministic eval Sharpe, then run stochastic on top-K

from pathlib import Path
import copy
import pandas as pd
import numpy as np

# ---- knobs ----
RESULTS_ROOT = Path("/content/tcn_tape_vectorized_version_clean/tcn_fusion_results")
HORIZON_DAYS = 756          # change to 252/504/756/1008
DET_SEED = 6042
TOP_K = 3
STOCH_RUNS_TOPK = 20        # set 0 to skip stochastic pass

# ---- resolve objects from notebook globals ----
exp_obj = globals().get("train_experiment6") or globals().get("experiment6")
phase1_obj = globals().get("eval_phase1_data") or globals().get("train_phase1_data")
base_cfg = globals().get("eval_config") or globals().get("train_config")

if exp_obj is None or phase1_obj is None or base_cfg is None:
    raise RuntimeError("Missing one of: experiment6/train_experiment6, eval_phase1_data/train_phase1_data, eval_config/train_config")

# if helper not in scope yet
if "evaluate_experiment6_checkpoint" not in globals():
    from src.notebook_helpers.tcn_phase1 import evaluate_experiment6_checkpoint

cfg = copy.deepcopy(base_cfg)

# ---- discover checkpoint prefixes ----
prefixes = []
for actor_path in (RESULTS_ROOT / "high_watermark_checkpoints").glob("*_actor.weights.h5"):
    prefix = str(actor_path).replace("_actor.weights.h5", "")
    critic_path = Path(prefix + "_critic.weights.h5")
    if critic_path.exists():
        prefixes.append(prefix)

prefixes = sorted(set(prefixes))
print(f"Found {len(prefixes)} candidate checkpoints")

# ---- deterministic ranking ----
det_rows = []
for i, prefix in enumerate(prefixes, 1):
    try:
        er = evaluate_experiment6_checkpoint(
            experiment6=exp_obj,
            phase1_data=phase1_obj,
            config=cfg,
            random_seed=DET_SEED,
            checkpoint_path_override=prefix,
            deterministic_eval_mode="mean",
            stochastic_eval_mode="sample",
            num_eval_runs=0,
            stochastic_episode_length_limit=HORIZON_DAYS,
            save_eval_logs=False,
            save_eval_artifacts=False,
        )
        dm = er.deterministic_metrics or {}
        det_rows.append({
            "prefix": prefix,
            "det_sharpe": float(dm.get("sharpe_ratio", np.nan)),
            "det_return_pct": float(dm.get("total_return_pct", np.nan)),
            "det_mdd_pct": float(dm.get("max_drawdown_pct", np.nan)),
            "det_turnover_pct": float(dm.get("turnover", np.nan)) * 100.0 if dm.get("turnover", None) is not None else np.nan,
        })
    except Exception as e:
        det_rows.append({"prefix": prefix, "error": f"{type(e).__name__}: {e}"})

det_df = pd.DataFrame(det_rows)
ok_df = det_df[det_df["det_sharpe"].notna()].copy()
ranked_df = ok_df.sort_values(["det_sharpe", "det_return_pct", "det_mdd_pct"], ascending=[False, False, True]).reset_index(drop=True)

print("\nTop deterministic checkpoints")
display(ranked_df.head(TOP_K))

# ---- stochastic pass on top-K ----
if STOCH_RUNS_TOPK > 0 and not ranked_df.empty:
    st_rows = []
    for _, r in ranked_df.head(TOP_K).iterrows():
        prefix = r["prefix"]
        er = evaluate_experiment6_checkpoint(
            experiment6=exp_obj,
            phase1_data=phase1_obj,
            config=cfg,
            random_seed=DET_SEED + 100000,
            checkpoint_path_override=prefix,
            deterministic_eval_mode="mean",
            stochastic_eval_mode="sample",
            num_eval_runs=STOCH_RUNS_TOPK,
            stochastic_episode_length_limit=HORIZON_DAYS,
            save_eval_logs=False,
            save_eval_artifacts=False,
        )
        sto = er.stochastic_results if isinstance(er.stochastic_results, pd.DataFrame) else pd.DataFrame()
        st_rows.append({
            "prefix": prefix,
            "det_sharpe": r["det_sharpe"],
            "sto_sharpe_mean": float(sto["sharpe_ratio"].mean()) if "sharpe_ratio" in sto else np.nan,
            "sto_sharpe_std": float(sto["sharpe_ratio"].std()) if "sharpe_ratio" in sto else np.nan,
            "sto_return_mean_pct": float(sto["total_return"].mean() * 100.0) if "total_return" in sto else np.nan,
            "sto_mdd_mean_pct": float(sto["max_drawdown"].mean() * 100.0) if "max_drawdown" in sto else np.nan,
        })

    st_df = pd.DataFrame(st_rows).sort_values(["sto_sharpe_mean", "det_sharpe"], ascending=[False, False]).reset_index(drop=True)
    print("\nTop-K stochastic rerank")
    display(st_df)